# Steel Industry Energy Forecasting: A Comparative Study of Temporal Encoding

Industrial power systems exhibit recurring intraday, weekly, and seasonal demand dynamics. Accurate short-term forecasting of electricity usage improves production planning, demand-side management, and operational cost efficiency. This study uses the Steel Industry Energy Consumption dataset to forecast hourly energy demand, with `Usage_kWh` as the target variable.

The project is structured as a controlled modeling experiment. We compare three regressors with different inductive biases: **Random Forest** and **XGBoost** (tree-based ensemble methods) versus **Linear Regression** (a parametric linear baseline). Each model is trained under two temporal feature representations: (1) **standard ordinal time features** (`hour`, `day_of_week`, `month`) and (2) **cyclic encodings** (sine/cosine transforms) that better preserve periodic continuity.

The core research objective is to quantify how model family and temporal encoding jointly influence forecast quality in hourly energy prediction.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

sns.set_theme(style="whitegrid", context="talk")
RANDOM_STATE = 42

data_path = ROOT / "Data" / "Steel_industry_data.csv"
df = pd.read_csv(data_path)

df["datetime"] = pd.to_datetime(df["date"], format="%d/%m/%Y %H:%M")
df = df.set_index("datetime").sort_index()

# Aggregate to hourly frequency to reduce intra-hour noise.
df_hourly = df[["Usage_kWh"]].resample("h").mean()

# Autoregressive lag features capture short-term and weekly recurrence patterns.
df_hourly["lag_1h"] = df_hourly["Usage_kWh"].shift(1)
df_hourly["lag_24h"] = df_hourly["Usage_kWh"].shift(24)
df_hourly["lag_168h"] = df_hourly["Usage_kWh"].shift(168)

df_hourly = df_hourly.dropna().copy()
df_hourly.head()


In [ ]:
feature_frame = df_hourly.copy()

# Standard (ordinal) time features.
feature_frame["hour"] = feature_frame.index.hour
feature_frame["day_of_week"] = feature_frame.index.dayofweek
feature_frame["month"] = feature_frame.index.month

lag_features = ["lag_1h", "lag_24h", "lag_168h"]
standard_time_features = ["hour", "day_of_week", "month"]
standard_features = standard_time_features + lag_features

# Cyclic encoding maps temporal boundaries smoothly (e.g., 23:00 near 00:00).
feature_frame["hour_sin"] = np.sin(2 * np.pi * feature_frame["hour"] / 24)
feature_frame["hour_cos"] = np.cos(2 * np.pi * feature_frame["hour"] / 24)
feature_frame["day_of_week_sin"] = np.sin(2 * np.pi * feature_frame["day_of_week"] / 7)
feature_frame["day_of_week_cos"] = np.cos(2 * np.pi * feature_frame["day_of_week"] / 7)
feature_frame["month_sin"] = np.sin(2 * np.pi * (feature_frame["month"] - 1) / 12)
feature_frame["month_cos"] = np.cos(2 * np.pi * (feature_frame["month"] - 1) / 12)

cyclic_time_features = [
    "hour_sin",
    "hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
]
cyclic_features = cyclic_time_features + lag_features

X_standard = feature_frame[standard_features]
X_cyclic = feature_frame[cyclic_features]
y = feature_frame["Usage_kWh"]

In [ ]:
split_idx = int(len(feature_frame) * 0.8)

X_std_train, X_std_test = X_standard.iloc[:split_idx], X_standard.iloc[split_idx:]
X_cyc_train, X_cyc_test = X_cyclic.iloc[:split_idx], X_cyclic.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "XGBoost": XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Linear Regression": LinearRegression(),
}

feature_sets = {
    "Standard": (X_std_train, X_std_test),
    "Cyclic": (X_cyc_train, X_cyc_test),
}


def train_and_evaluate(model_name, model, X_train, X_test, y_train, y_test, encoding):
    """Train a single model/encoding pair and return metrics."""
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    return {
        "Model": model_name,
        "Encoding": encoding,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": rmse,
        "R2": r2_score(y_test, predictions),
    }


from sklearn.base import clone

results = []
fitted_models = {}

for encoding_name, (X_train_set, X_test_set) in feature_sets.items():
    for model_name, model_template in models.items():
        model = clone(model_template)
        metrics = train_and_evaluate(
            model_name=model_name,
            model=model,
            X_train=X_train_set,
            X_test=X_test_set,
            y_train=y_train,
            y_test=y_test,
            encoding=encoding_name,
        )
        results.append(metrics)
        fitted_models[(model_name, encoding_name)] = {
            "model": model,
            "features": X_train_set.columns.tolist(),
        }

In [ ]:
def extract_feature_importance(model, feature_names):
    """Return normalized feature importance for tree models or linear coefficients."""
    if hasattr(model, "feature_importances_"):
        values = model.feature_importances_
        method = "feature_importances_"
    else:
        values = np.abs(model.coef_)
        method = "abs(coef_)"

    importance = pd.DataFrame(
        {"feature": feature_names, "importance": values}
    ).sort_values("importance", ascending=False)

    importance["importance_pct"] = (
        100 * importance["importance"] / importance["importance"].sum()
    )
    importance["method"] = method
    return importance


importance_rows = []
for (model_name, encoding_name), fitted in fitted_models.items():
    imp = extract_feature_importance(
        fitted["model"], fitted["features"]
    )
    imp["Model"] = model_name
    imp["Encoding"] = encoding_name
    importance_rows.append(imp)

importance_df = pd.concat(importance_rows, ignore_index=True)

# Wide view: one row per feature, columns per model/encoding combination.
importance_pivot = importance_df.pivot_table(
    index="feature",
    columns=["Model", "Encoding"],
    values="importance_pct",
    aggfunc="first",
).round(2)

display(importance_pivot.style.background_gradient(cmap="YlOrRd", axis=None))

# Horizontal bar charts for each trained model.
plot_order = [
    (model_name, encoding_name)
    for encoding_name in feature_sets
    for model_name in models
]

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10), constrained_layout=True)
axes = axes.ravel()

for axis, (model_name, encoding_name) in zip(axes, plot_order):
    subset = importance_df[
        (importance_df["Model"] == model_name)
        & (importance_df["Encoding"] == encoding_name)
    ].sort_values("importance", ascending=True)

    axis.barh(subset["feature"], subset["importance_pct"], color="#4C72B0")
    axis.set_title(f"{model_name} | {encoding_name}")
    axis.set_xlabel("Importance (%)")
    axis.set_ylabel("Feature")

plt.suptitle("Feature Importance by Model and Temporal Encoding", y=1.02)
plt.show()

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["Model", "Encoding"]).reset_index(drop=True)

styled_results = results_df.style.format(
    {"MAE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.4f}"}
).background_gradient(cmap="YlGnBu", subset=["MAE", "RMSE", "R2"])

styled_results

In [ ]:
viz_df = results_df.copy()
palette = "Set2"

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 6), constrained_layout=True)

sns.barplot(
    data=viz_df,
    x="Model",
    y="MAE",
    hue="Encoding",
    palette=palette,
    ax=axes[0],
)
axes[0].set_title("MAE Comparison: Standard vs Cyclic Temporal Encoding")
axes[0].set_xlabel("Model")
axes[0].set_ylabel("Mean Absolute Error")
axes[0].legend(title="Feature Encoding")

sns.barplot(
    data=viz_df,
    x="Model",
    y="R2",
    hue="Encoding",
    palette=palette,
    ax=axes[1],
)
axes[1].set_title("R² Comparison: Standard vs Cyclic Temporal Encoding")
axes[1].set_xlabel("Model")
axes[1].set_ylabel("R² Score")
axes[1].legend(title="Feature Encoding")

for axis in axes:
    axis.tick_params(axis="x", rotation=10)

plt.show()

## Key Insights and Conclusion

The comparative experiment is expected to show that autoregressive memory is central to performance in this dataset. In particular, `lag_1h` captures short-horizon inertia and immediate process continuity, while `lag_168h` captures weekly operational periodicity; together, these are typically among the strongest predictors for hourly industrial demand.

Tree-based models (Random Forest and XGBoost) are expected to remain highly competitive under standard ordinal time variables because they naturally model nonlinear thresholds and interactions in raw temporal partitions. By contrast, Linear Regression has limited capacity to represent periodicity from ordinal time variables alone and therefore tends to benefit the most from cyclic encodings, where sine/cosine transformations provide a geometrically consistent representation of time.

Overall, this framework provides a reproducible and research-aligned benchmark for evaluating how feature representation influences model behavior in energy forecasting, while maintaining a transparent comparison across model families.